# 03 — Churn Prediction + Revenue at Risk
**Model:** Logistic Regression | **Goal:** Business usefulness, not just accuracy.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import json

features = pd.read_csv('../processed/features.csv')
print(f"Dataset: {len(features)} customers | Churn rate: {features['churned'].mean():.1%}")


## Train Logistic Regression

In [ ]:
MODEL_FEATURES = [
    'days_since_last_purchase', 'total_orders', 'avg_order_value',
    'purchase_frequency', 'customer_lifespan_days', 'avg_days_between_orders',
    'unique_products', 'r_score', 'f_score', 'm_score', 'days_since_signup'
]

X = features[MODEL_FEATURES].fillna(0)
y = features['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_sc, y_train)
print("Model trained!")


## Evaluate Model

In [ ]:
y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

# Feature importance
coef_df = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print("\nTop drivers of churn:")
print(coef_df.to_string(index=False))


## Revenue at Risk

In [ ]:
X_all = scaler.transform(features[MODEL_FEATURES].fillna(0))
features['churn_prob'] = model.predict_proba(X_all)[:, 1]
features['revenue_at_risk'] = (features['churn_prob'] * features['ltv']).round(2)

threshold = features['revenue_at_risk'].quantile(0.80)
features['is_top_20_risk'] = (features['revenue_at_risk'] >= threshold).astype(int)

top_risk = features[features['is_top_20_risk'] == 1]

print(f"Total Revenue at Risk:  ₹{features['revenue_at_risk'].sum():,.0f}")
print(f"Top 20% customers RAR:  ₹{top_risk['revenue_at_risk'].sum():,.0f}")
print(f"Customers to target:    {len(top_risk)}")


## Decision Engine

In [ ]:
CAMPAIGN_SUCCESS = 0.25

revenue_saveable = top_risk['revenue_at_risk'].sum() * CAMPAIGN_SUCCESS
print(f"\n💡 DECISION ENGINE OUTPUT")
print(f"Revenue Saveable (25% campaign):  ₹{revenue_saveable:,.0f}")
print(f"Customers to campaign:            {len(top_risk)}")
print(f"Avg Revenue at Risk per customer: ₹{top_risk['revenue_at_risk'].mean():,.0f}")

print("\nSegment breakdown of top 20% risk:")
print(top_risk.groupby('rfm_segment')['revenue_at_risk'].agg(['count','sum','mean']).round(0))


## Save Outputs

In [ ]:
def recommend_action(row):
    seg, prob = row.get('rfm_segment',''), row['churn_prob']
    if seg in ['Champions','Loyal Customers'] and prob > 0.6: return 'VIP Re-engagement Campaign'
    if seg in ['At Risk','Cant Lose Them']: return 'Urgent Win-back Campaign'
    if seg == 'Hibernating': return 'Reactivation Email Series'
    return 'Discount + Personalized Outreach'

features['recommended_action'] = features.apply(recommend_action, axis=1)

churn_scores = features[['customer_id','churn_prob','ltv','revenue_at_risk',
                          'is_top_20_risk','rfm_segment','recommended_action',
                          'days_since_last_purchase','total_orders','avg_order_value']]
churn_scores.to_csv('../processed/churn_scores.csv', index=False)
print("✅ churn_scores.csv saved")
